In [8]:
import sys
import os
from pathlib import Path

# Add project root to sys.path
try:
    notebook_dir = Path(os.getcwd()).resolve()
    project_root = notebook_dir.parents[3]
except Exception:
    project_root = Path("../../../../").resolve()

sys.path.append(str(project_root))

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

from src.infrastructure.dataset.dataset_factory import DatasetFactory
from src.domain.model.proactive_forest import ProactiveForest
from src.infrastructure.models.random_forest_adapter import RandomForestWrapper
from src.domain.metrics.forest_evaluator import ForestEvaluator
from src.infrastructure.metrics.sklearn_metrics_service import SklearnMetricsService

# Initialize services
metrics_svc = SklearnMetricsService()
print(f"Project Root identified: {project_root}")

Project Root identified: C:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest


# Centralized Baselines Comparison

This notebook evaluates the performance of a classic Random Forest and a Proactive Forest on 8 different datasets in a centralized (non-federated) context.

## Metrics
- **Accuracy**: Standard classification accuracy.
- **PCD (Percentage Correct Diversity)**: Percentage of samples where 10% to 90% of the trees are correct. This follows the project's official definition (Cepero, 2023).
- **F1 Score**: Macro-averaged F1 score.

In [9]:
datasets = [
    "car", "Iris", "letter", "nursery", 
    "optdigits", "sonar", "spambase", "vowel"
]

results = []

for dataset_name in tqdm(datasets, desc="Evaluating Datasets"):
    try:
        # Load dataset
        # Note: DatasetFactory.load_from_config expects the dataset config directly
        cfg = {"type": dataset_name, "test_size": 0.2, "scale": True}
        split = DatasetFactory.load_from_config(cfg, project_root=project_root)
        
        # 1. Train Random Forest (Classic)
        rf = RandomForestWrapper(n_estimators=100, random_state=42)
        rf.fit(split.X_train, split.y_train)
        
        # 2. Train Proactive Forest
        pf = ProactiveForest(n_estimators=100, alpha=0.1, random_state=42, class_names=split.class_names)
        pf.fit(split.X_train, split.y_train)
        
        # Standard Evaluation (Accuracy, F1, PCD)
        pf_report = ForestEvaluator.evaluate(pf, split.X_test, split.y_test, split.class_names, metrics_svc)
        rf_report = ForestEvaluator.evaluate(rf, split.X_test, split.y_test, split.class_names, metrics_svc)
        
        # Store results
        results.append({
            "Dataset": dataset_name.capitalize(),
            "Model": "Random Forest",
            "Accuracy": rf_report.accuracy,
            "PCD": rf_report.pcd,
            "F1 Score": rf_report.macro_f1
        })
        
        results.append({
            "Dataset": dataset_name.capitalize(),
            "Model": "Proactive Forest",
            "Accuracy": pf_report.accuracy,
            "PCD": pf_report.pcd,
            "F1 Score": pf_report.macro_f1
        })
    except Exception as e:
        print(f"Error processing dataset {dataset_name}: {e}")
        continue

Evaluating Datasets:   0%|          | 0/8 [00:00<?, ?it/s]

In [10]:
# Display results table
if results:
    df_results = pd.DataFrame(results)
    df_pivot = df_results.pivot(index="Dataset", columns="Model", values=["Accuracy", "PCD", "F1 Score"])
    
    # Styled display
    display(df_pivot.style.highlight_max(axis=1, color='lightgreen'))
    
    # Save results
    output_dir = project_root / "results"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "baselines_comparison.csv"
    
    df_results.to_csv(output_path, index=False)
    print(f"Results saved to: {output_path.absolute()}")
else:
    print("No results to display. Check for errors in the evaluation loop.")

Results saved to: C:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\baselines_comparison.csv
